# SIB - RidgeRegression (Gradient Descent + L2)

Teste da classe `RidgeRegression` com o dataset `cpu.csv` (problema de regressão).

O modelo assume uma relação linear entre as features e a variável dependente:

$$h_\theta(x) = \theta^T x = \theta_0 x_0 + \theta_1 x_1 + \dots + \theta_n x_n$$

e estima os coeficientes por **gradient descent** com regularização **L2**, minimizando

$$J(\theta) = \frac{1}{2m} \left[ \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2 + \lambda \sum_{j=1}^{n} \theta_j^2 \right]$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from si.io.csv_file import read_csv
from si.metrics.mse import mse
from si.model_selection.split import train_test_split
from si.models.linear_regression import RidgeRegression

## 1. mse

Erro quadrático médio entre os valores reais e previstos de y.

In [2]:
y_true = np.array([1.0, 2.0, 3.0, 4.0])
y_pred = np.array([1.0, 2.0, 3.0, 6.0])

print('mse:', mse(y_true, y_pred))          # (0 + 0 + 0 + 4) / 4 = 1.0
print('mse (previsão perfeita):', mse(y_true, y_true))

mse: 1.0
mse (previsão perfeita): 0.0


## 2. Dataset cpu.csv

209 CPUs; queremos estimar a performance relativa (`perf`).

In [3]:
cpu = read_csv('../datasets/cpu/cpu.csv', sep=',', features=True, label=True)

print('dimensões:', cpu.shape())
print('features:', cpu.features)
print('label:', cpu.label)
cpu.summary()

dimensões: (209, 6)
features: ['syct', 'mmin', 'mmax', 'cach', 'chmin', 'chmax']
label: perf


,syct,mmin,mmax,cach,chmin,chmax
mean,203.822967,2.867981e+03,1.179615e+04,25.205742,4.698565,18.267943
median,110.000000,2.000000e+03,8.000000e+03,8.000000,2.000000,8.000000
min,17.000000,6.400000e+01,6.400000e+01,0.000000,0.000000,0.000000
max,1500.000000,3.200000e+04,6.400000e+04,256.000000,52.000000,176.000000
var,67412.691147,1.497266e+07,1.368544e+08,1642.794991,46.239280,672.626771


## 3. Divisão em treino e teste

In [4]:
train, test = train_test_split(cpu, test_size=0.2, random_state=42)

print('treino:', train.shape())
print('teste: ', test.shape())

treino: (168, 6)
teste:  (41, 6)


## 4. Treino do modelo

As features têm escalas muito diferentes (`syct` ~ 10² vs `mmax` ~ 10⁴), por isso `scale=True`
é essencial: sem normalização o gradient descent diverge para qualquer alpha razoável.

In [5]:
model = RidgeRegression(l2_penalty=1.0, alpha=0.01, max_iter=10000, patience=100, scale=True)
model.fit(train)

print('iterações efetuadas:', len(model.cost_history), '(max_iter = 10000)')
print('theta_zero: {:.4f}'.format(model.theta_zero))
for feature, coef in zip(train.features, model.theta):
    print('  {:<6s} {:>9.4f}'.format(feature, coef))

iterações efetuadas: 1530 (max_iter = 10000)
theta_zero: 99.0595
  syct     11.2648
  mmin     37.2419
  mmax     63.7882
  cach     25.6921
  chmin     2.7450
  chmax    32.2045


## 5. Score e custo

`score` devolve o **mse**; `cost` devolve a função de custo J (mse/2 + termo L2).

In [6]:
print('mse  (treino): {:.4f}'.format(model.score(train)))
print('mse  (teste):  {:.4f}'.format(model.score(test)))
print('cost (treino): {:.4f}'.format(model.cost(train)))
print('cost (teste):  {:.4f}'.format(model.cost(test)))

mse  (treino): 3102.3462
mse  (teste):  5790.8423
cost (treino): 1572.8622
cost (teste):  2984.2936


## 6. Convergência

O custo tem de descer monotonicamente. Se subir, o `alpha` é demasiado alto.

In [7]:
iterations = list(model.cost_history.keys())
costs = list(model.cost_history.values())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(iterations, costs)
axes[0].set(xlabel='iteração', ylabel='custo J', title='Função de custo')
axes[1].plot(iterations[:200], costs[:200])
axes[1].set(xlabel='iteração', ylabel='custo J', title='Primeiras 200 iterações')
plt.tight_layout()
plt.show()

print('custo inicial: {:.4f}'.format(costs[0]))
print('custo final:   {:.4f}'.format(costs[-1]))

custo inicial: 13900.5219
custo final:   1572.8622


## 7. Efeito do l2_penalty e do alpha

A penalização L2 encolhe os coeficientes: quanto maior o lambda, menor a soma dos |theta|,
ao custo de mais enviesamento.

In [8]:
print('{:>12s} {:>12s} {:>12s} {:>12s}'.format('l2_penalty', 'sum|theta|', 'mse treino', 'mse teste'))
for l2 in [0.0, 1.0, 10.0, 100.0, 1000.0]:
    m = RidgeRegression(l2_penalty=l2, alpha=0.01, max_iter=10000, patience=100, scale=True).fit(train)
    print('{:>12.1f} {:>12.4f} {:>12.2f} {:>12.2f}'.format(
        l2, np.sum(np.abs(m.theta)), m.score(train), m.score(test)))

  l2_penalty   sum|theta|   mse treino    mse teste
         0.0     173.6225      3102.03      5760.69
         1.0     172.9365      3102.35      5790.84
        10.0     167.1422      3123.74      6097.90
       100.0     130.4375      3877.64      9730.99
      1000.0      56.0653     10429.56     30069.52


In [9]:
print('{:>8s} {:>10s} {:>14s} {:>12s}'.format('alpha', 'iterações', 'custo final', 'mse teste'))
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    m = RidgeRegression(l2_penalty=1.0, alpha=alpha, max_iter=10000, patience=100, scale=True).fit(train)
    print('{:>8.4f} {:>10d} {:>14.4f} {:>12.2f}'.format(
        alpha, len(m.cost_history), list(m.cost_history.values())[-1], m.score(test)))

   alpha  iterações    custo final    mse teste


  0.0001      10000      2506.4923      9975.75


  0.0010      10000      1573.0235      5770.12
  0.0100       1530      1572.8622      5790.84
  0.1000        297      1572.8464      5805.17


## Conclusões

- O `scale=True` é obrigatório neste dataset: as features diferem em várias ordens de grandeza.
- O `patience` evita gastar as 10000 iterações quando o custo já estabilizou.
- Aumentar o `l2_penalty` encolhe os coeficientes; a partir de um certo ponto o mse de teste piora
  (o modelo fica demasiado enviesado).